# Conditional Randomization Test (CRT) Calibration

Mirrors `Slurm_version/CRT/CRT.py` for interactive use.

CRT tests the differential effect of each gene target on each program, with optional skew-normal calibration. NTC (non-targeting) guides are randomly grouped into ensembles and used as the calibration null.

## Output structure

```
{out_dir}/{run_name}/Evaluation/{K}_{thresh}/
├── {K}_CRT_{covar_tag}_{condition}.txt    # target × program p-values + log2FC + adj_pval (per condition)
└── {K}_CRT_{covar_tag}_{condition}.png    # QQ plot: grouped NTC controls (raw vs skew) vs CRT null
```

`{covar_tag}` joins the `covariates` and `log_covariates` lists with `_`, prefixing log-transformed ones with `log_` (matching `.py`'s `_covariate_tag`). When no covariates are set, the tag is `no_covariates`.

## Assumptions about the input `.h5mu`

The `.h5mu` produced by Stage 1 inference already contains, inside the `prog_key` (`"cNMF"`) modality:
- `obsm["guide_assignment"]` (cells × guides)
- `uns["guide_names"]` (list of guide names matching guide_assignment columns)
- `uns["guide_targets"]` (target gene per guide)

So there is no separate `mdata_guide` file — the notebook reads exactly the same path that `.py` does.


In [ ]:
import os
import sys
import yaml

import numpy as np
import pandas as pd
import muon as mu
import scanpy as sc
import matplotlib.pyplot as plt

# Change path to wherever you have the repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from tests.synthetic_data import make_sceptre_style_synth  # noqa: F401 — exposed for ad-hoc synthetic data testing
from src.sceptre import (
    prepare_crt_inputs,
    build_ntc_group_inputs,
    compute_guide_set_null_pvals,
    crt_pvals_for_ntc_groups_ensemble,
    crt_pvals_for_ntc_groups_ensemble_skew,
    make_ntc_groups_ensemble,
    run_all_genes_union_crt,
)
from src.visualization import qq_plot_ntc_pvals


## Step 1. Set Up

In [ ]:
# ── IO ──
out_dir = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result"
run_name = "020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50"

# K and density thresholds to test
K = [50]
sel_threshs = [0.2]

# Optional: override the per-(K, thresh) output base directory. None ⇒ {out_dir}/{run_name}/Evaluation/.
save_dir = None

# ── MuData keys ──
categorical_key = "timepoint"   # obs column with cell condition/sample labels

# ── Covariates (passed straight into reformat_data_for_CRT) ──
covariates = ["biological_sample"]
log_covariates = ["guide_umi_counts", "n_genes_by_counts", "total_counts", "pct_counts_mt"]

# ── Calibration parameters ──
number_guide = 6                           # NTC group size
number_permutations = 1024                 # CRT B
guide_annotation_key = ["non-targeting"]   # NTC target group name(s) in mdata['cNMF'].uns['guide_targets']
FDR_method = "BH"                          # "BH" or "StoreyQ"


## Step 2. Reformat MuData for CRT

Reads guide info directly from `mdata["cNMF"]` — no separate guide AnnData required.


In [ ]:
def reformat_data_for_CRT(mdata, covariates=None, log_covariates=None):
    adata = mdata["cNMF"].copy()
    adata.obsm["cnmf_usage"] = np.asarray(adata.X)  # ensure dense float

    # Guide assignment already lives in the cNMF modality of the input mdata
    adata.obsm["guide_assignment"] = adata.obsm["guide_assignment"].copy()

    # Program names
    adata.uns["program_names"] = list(adata.var_names)

    # Guide names must match columns of guide_assignment
    guide_names = list(adata.uns["guide_names"])
    adata.uns["guide_names"] = guide_names

    guide2gene = dict(zip(guide_names, adata.uns["guide_targets"]))
    adata.uns["guide2gene"] = guide2gene

    # Covariates
    covar_dict = {}
    if covariates:
        for key in covariates:
            covar_dict[key] = adata.obs[key]
    if log_covariates:
        for key in log_covariates:
            covar_dict[f"log_{key}"] = np.log1p(adata.obs[key])

    adata.obsm["covar"] = pd.DataFrame(covar_dict, index=adata.obs_names)

    return adata


## Step 3. CRT helpers

`run_CRT` performs CRT per condition (iterates over `adata.obs[categorical_key].unique()` — no more hardcoded `'d3'`). `save_result` writes the long-form `target × program × log2FC × p-value × adj_pval` table.


In [ ]:
def _covariate_tag(covariates, log_covariates):
    """Build filename token from covariate lists, e.g. biological_sample_log_guide_umi_counts."""
    parts = list(covariates or []) + [f"log_{c}" for c in (log_covariates or [])]
    return "_".join(parts) if parts else "no_covariates"


def save_result(out, k, output_folder, condition, covar_tag, FDR_method="BH"):
    """Melt skew-calibrated pvals + betas to long form, FDR-correct, write to disk."""

    pval_df = out["pvals_skew_df"]
    beta_df = out["betas_df"]

    pval_long = pval_df.reset_index().melt(
        id_vars="index", var_name="program_name", value_name="p-value"
    ).rename(columns={"index": "target_name"})

    beta_long = beta_df.reset_index().melt(
        id_vars="index", var_name="program_name", value_name="log2FC"
    ).rename(columns={"index": "target_name"})

    result_df = pval_long.merge(beta_long, on=["program_name", "target_name"], how="inner")
    result_df = result_df[["target_name", "program_name", "log2FC", "p-value"]]

    # Note: CRT's beta is not exactly log2FC. The transform is approx_log2FC = [K / (K - 1)] * beta_hat / ln(2).
    # We keep the raw beta as-is (matches .py).

    if FDR_method == "BH":
        from statsmodels.stats.multitest import multipletests
        result_df["adj_pval"] = multipletests(result_df["p-value"], method="fdr_bh")[1]
    elif FDR_method == "StoreyQ":
        from multipy.fdr import qvalue
        import builtins
        if not hasattr(builtins, "xrange"):
            builtins.xrange = range
        result_df["adj_pval"] = qvalue(result_df["p-value"].values, threshold=0.05, verbose=False)[1]
    else:
        raise ValueError(f"Unknown FDR_method: {FDR_method}")

    result_df.to_csv(
        f"{output_folder}/{k}_CRT_{covar_tag}_{condition}.txt", sep="\t", index=False
    )
    return result_df


def run_CRT(adata, k, sel_thresh, output_folder,
            categorical_key, guide_annotation_key, number_guide, number_permutations,
            FDR_method, covariates, log_covariates):
    """Run CRT for each condition of cells in adata[categorical_key]."""

    covar_tag = _covariate_tag(covariates, log_covariates)

    for condition in adata.obs[categorical_key].unique():
        print(f"  Condition: {condition}")
        adata_con = adata[adata.obs[categorical_key] == condition].copy()

        inputs = prepare_crt_inputs(
            adata=adata_con,
            usage_key="cnmf_usage",
            covar_key="covar",
            guide_assignment_key="guide_assignment",
            guide2gene_key="guide2gene",
        )

        out = run_all_genes_union_crt(
            inputs=inputs,
            B=number_permutations,
            n_jobs=-1,
            calibrate_skew_normal=True,
            return_raw_pvals=True,
            return_skew_normal=True,
        )

        # NTC-grouped null
        ntc_labels = guide_annotation_key
        ntc_guides, guide_freq, guide_to_bin, real_sigs = build_ntc_group_inputs(
            inputs=inputs,
            ntc_label=ntc_labels,
            group_size=number_guide,
            n_bins=10,
        )

        ntc_groups_ens = make_ntc_groups_ensemble(
            ntc_guides=ntc_guides,
            ntc_freq=guide_freq,
            real_gene_bin_sigs=real_sigs,
            guide_to_bin=guide_to_bin,
            n_ensemble=10,
            seed0=7,
            group_size=number_guide,
            max_groups=None,
        )

        ntc_group_pvals_ens = crt_pvals_for_ntc_groups_ensemble(
            inputs=inputs,
            ntc_groups_ens=ntc_groups_ens,
            B=number_permutations,
            seed0=23,
        )

        ntc_group_pvals_skew_ens = crt_pvals_for_ntc_groups_ensemble_skew(
            inputs=inputs,
            ntc_groups_ens=ntc_groups_ens,
            B=number_permutations,
            seed0=23,
        )

        # CRT-null matched to NTC group units
        guide_to_col = {g: i for i, g in enumerate(inputs.guide_names)}
        null_pvals = np.concatenate(
            [
                compute_guide_set_null_pvals(
                    guide_idx=[guide_to_col[g] for g in guides],
                    inputs=inputs,
                    B=number_permutations,
                ).ravel()
                for groups in ntc_groups_ens
                for guides in groups.values()
            ]
        )

        ax = qq_plot_ntc_pvals(
            pvals_raw_df=out["pvals_raw_df"],
            guide2gene=adata.uns["guide2gene"],
            ntc_genes=ntc_labels,
            pvals_skew_df=out["pvals_df"],
            null_pvals=null_pvals,
            ntc_group_pvals_ens=ntc_group_pvals_ens,
            ntc_group_pvals_skew_ens=ntc_group_pvals_skew_ens,
            show_ntc_ensemble_band=True,
            show_all_pvals=True,
            title=f"QQ plot: grouped NTC controls (raw vs skew) vs CRT null for {condition}",
        )

        plt.tight_layout()
        plt.savefig(f"{output_folder}/{k}_CRT_{covar_tag}_{condition}.png", dpi=100)
        save_result(out, k, output_folder, condition, covar_tag, FDR_method=FDR_method)
        plt.close()


## Step 4. Main loop

In [ ]:
for sel_thresh in sel_threshs:
    thresh_tag = str(sel_thresh).replace(".", "_")
    for k in K:
        print(f"Processing K={k}, sel_thresh={sel_thresh}")

        if save_dir:
            output_folder = f"{save_dir}/{k}_{thresh_tag}"
        else:
            output_folder = f"{out_dir}/{run_name}/Evaluation/{k}_{thresh_tag}"
        os.makedirs(output_folder, exist_ok=True)

        # Load mdata — matches .py path (Inference/adata)
        mdata = mu.read(
            f"{out_dir}/{run_name}/Inference/adata/cNMF_{k}_{thresh_tag}.h5mu"
        )

        # Build CRT-ready adata
        adata = reformat_data_for_CRT(
            mdata, covariates=covariates, log_covariates=log_covariates
        )

        # Floor program-usage matrix to avoid divide-by-zero rows
        U = adata.obsm["cnmf_usage"].copy()
        U = np.maximum(U, 1e-8)
        U /= U.sum(axis=1, keepdims=True)
        adata.obsm["cnmf_usage"] = U

        # Run CRT per condition
        run_CRT(
            adata,
            k=k,
            sel_thresh=sel_thresh,
            output_folder=output_folder,
            categorical_key=categorical_key,
            guide_annotation_key=guide_annotation_key,
            number_guide=number_guide,
            number_permutations=number_permutations,
            FDR_method=FDR_method,
            covariates=covariates,
            log_covariates=log_covariates,
        )

print("Pipeline finished.")
